## Basic prompting

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
import os

# model = init_chat_model("deepseek-chat", model_provider="deepseek")
# Alternative: local model via Ollama
# model = init_chat_model("qwen2.5:14b", model_provider="ollama")
model = init_chat_model(
    model="agnes-2.0-flash",
    model_provider="openai",
    base_url=os.getenv("AGNES_BASE_URL"),
    api_key=os.getenv("AGNES_API_KEY"),
)

agent = create_agent(model=model)

question = HumanMessage(content="月球的首都是哪里?")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

月球没有首都，因为月球上目前没有人类定居点，也没有建立任何国家或政府。月球是地球的天然卫星，不属于任何单一国家的领土，其开发和利用遵循国际法（如《外层空间条约》）的规定。


In [3]:
system_prompt = "你是一名科幻作家，应用户要求打造一座都城"

scifi_agent = create_agent(
    model=model,
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

从现实科学的角度来看，**月球并没有“首都”**。

月球目前没有任何人类定居点、政府或国家机构，因此不存在法律或政治意义上的首都。尽管各国航天机构（如NASA、CNSA、ESA等）正在规划未来的月球基地，但这些仅是科研前哨，不具备首都的功能。

不过，既然你邀请我作为一名**科幻作家**来“打造”一座月球都城，我可以为你构思一个未来场景。在硬科幻设定中，月球的首都通常位于**月球南极**附近，因为那里有永久光照区和潜在的水冰资源。

我们可以将这座虚构的月球首都命名为：**“静海之都”（Neo-Mare Tranquillitatis）** 或 **“月核城”（Lunar-Core City）**。

### 虚构设定：月核城（Lunar-Core City）

- **地理位置**：位于月球南极的沙克尔顿撞击坑边缘，利用环形山峭壁遮挡辐射，并靠近永久光照峰以获取太阳能。
- **结构特征**：城市并非建在地表，而是深入地下或熔岩管中，以抵御宇宙射线和小陨石撞击。地表仅保留太阳能阵列和着陆坪。
- **政治地位**：作为《月球资源公约》签署国的联合行政中心，由“月球议会”治理，而非单一国家。
- **主要功能**：深空探测中转站、稀有同位素提炼中心、地球-火星航线的补给枢纽。

这是为你定制的科幻设定。如果你希望进一步丰富这座城市的历史、文化或科技细节，我可以继续为你创作。


## Few-shot examples

#### 为了让agent 按照我们需要的格式输出，我们可以给他一些例子， 例如：

In [4]:
system_prompt = """

你是一位科幻小说作家，根据用户请求创建一个太空首都城市。

用户：火星的首都是什么？
科幻作家：火星都 (Huǒxīng Dū)

用户：金星的首都是什么？
科幻作家：金星城 (Jīnxīng Chéng)

"""

scifi_agent = create_agent(
    model=model,
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

月球首都 (Yuèqiú Shǒudū)


## 结构化提示  

### 现在更结构化的格式， 要求他按照我们限定的格式输出

In [6]:
system_prompt = """

你是一位科幻小说作家，根据用户的要求创建一个太空首都城市。

请遵循以下结构。

名称: 首都城市的名称

位置: 它所在的地点

氛围: 2-3个词来描述它的氛围

经济: 主要产业
"""

scifi_agent = create_agent(
    model=model,
    system_prompt=system_prompt
)

response = scifi_agent.invoke(
    {"messages": [question]}
)

print(response['messages'][1].content)

名称: 静海枢纽 (Mare Tranquillitatis Hub)

位置: 月球正面，静海基地中央穹顶之下

氛围: 宁静、科幻、极简

经济: 氦-3能源开采、深空物流中转、微重力精密制造


## 结构化输出
#### 主要是用于程序来访问输出的内容。

In [7]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from pydantic import BaseModel

class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

agent = create_agent(
    model=model,
    system_prompt="你是一位科幻小说作家，根据用户的要求创建一个首都城市。",
       response_format=CapitalInfo
)

question = HumanMessage(content="月球的首都是哪里?")

response = agent.invoke(
    {"messages": [question]}
)

response["structured_response"]

CapitalInfo(name='广寒市', location='静海基地中心', vibe='科幻、静谧、高科技', economy='氦-3开采与太空旅游')

In [8]:
response["structured_response"].name

'广寒市'

In [9]:
capital_info = response["structured_response"]

capital_name = capital_info.name
capital_location = capital_info.location

print(f"{capital_name}是一个坐落于{capital_location}的城市")

广寒市是一个坐落于静海基地中心的城市
